# 01 Exposure Model and Inhaled Aerosol Distribution

## Purpose

This notebook reproduces the source-to-recipient aerosol exposure calculation used in the manuscript.

It derives:

- the expected number of aerosols inhaled during the defined exposure period, \(\mu\);
- the inhaled aerosol-radius probability density, \(f_R(r)\).

The calculation uses the emitted aerosol-radius distribution, ventilation removal, size-dependent aerosol deposition/removal, exposure duration, room volume, aerosol emission rate, and recipient inhalation rate.

This notebook does not generate infection outcomes or fit the Bayesian calibration or in vivo models.

In [ ]:
# Fixed exposure-model quantities
import numpy as np

p = 0.0115          # recipient inhalation rate, m^3/min
E_A = 600.0         # aerosol emission rate, aerosols/min
V = 100.0           # room volume, m^3
tau = 240.0         # exposure duration, min

kappa_vent = 0.00833   # ventilation removal rate, min^-1

# Modelled aerosol-radius range
r_min = 0.15        # micrometres
r_max = 2.50        # micrometres

# Emitted aerosol-radius distribution
r_median = 0.139    # micrometres
GSD = 1.64

mu_emit = np.log(r_median)
sigma_emit = np.log(GSD)

print("Exposure-model settings loaded.")

Exposure-model settings loaded.


In [ ]:
def f_emit(r):
    """
    Emitted aerosol-radius probability density.
    Radius r is in micrometres.
    """
    r = np.asarray(r)
    return (
        1.0
        / (r * sigma_emit * np.sqrt(2.0 * np.pi))
        * np.exp(
            -((np.log(r) - mu_emit) ** 2)
            / (2.0 * sigma_emit**2)
        )
    )


def kappa_dep(r):
    """
    Size-dependent aerosol deposition/removal rate in min^-1.
    Radius r is in micrometres.
    """
    r = np.asarray(r)

    return np.select(
        [
            (r >= 0.15) & (r < 0.25),
            (r >= 0.25) & (r < 0.35),
            (r >= 0.35) & (r < 0.50),
            (r >= 0.50) & (r <= 2.50),
        ],
        [
            0.00567,
            0.0100,
            0.0125,
            0.0172,
        ],
        default=np.nan,
    )


def kappa_A(r):
    """
    Total aerosol removal rate in min^-1.
    Ventilation and size-dependent deposition/removal are included.
    """
    return kappa_vent + kappa_dep(r)


# Numerical grid over the modelled radius range
r_grid = np.linspace(r_min, r_max, 50000)

kappa_grid = kappa_A(r_grid)

# Unnormalised exposure-weighted aerosol-radius density
exposure_weight = (
    f_emit(r_grid)
    / kappa_grid
    * (
        tau
        - (1.0 - np.exp(-kappa_grid * tau))
        / kappa_grid
    )
)

# Integral of the unnormalised exposure weight
normalising_constant = np.trapezoid(
    exposure_weight,
    r_grid
)

# Inhaled aerosol-radius probability density
f_R_grid = exposure_weight / normalising_constant

# Expected number of inhaled aerosols
mu = (
    p
    * E_A
    / V
    * normalising_constant
)

print(f"Derived mean inhaled aerosol number mu = {mu:.3f}")
print(f"Numerical integral of f_R(r) = {np.trapezoid(f_R_grid, r_grid):.6f}")

Derived mean inhaled aerosol number mu = 350.757
Numerical integral of f_R(r) = 1.000000


## Expected baseline result

With the fixed exposure settings above, the calculation gives

\[
\mu \approx 350.76,
\]

which is reported approximately as \(\mu=350\) in the manuscript simulation settings.

The numerical integral of \(f_R(r)\) over the modelled radius range should equal 1.